In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import tarfile
import pandas as pd
import pyarrow
import pandera.pandas as pa
import numpy as np
import duckdb
import os
import sys
sys.path.append(str(Path(os.getcwd()).parent.absolute()))
import colorlog
logger = colorlog.getLogger()

from ringer_zero.root import read_ttree_as_pdf
TTREE_NAME = 'run_410000/HLT/EgammaMon/summary/events'

In [43]:
source_files = {
    'jf17': {
        'datapath': Path(
            '/media/lucasbanunes/KINGSTON/data/cern-datasets/mc21_isabela_qt_no_restriction/source/bkg/user.isilvafe.mc21_13p6TeV.JF17.AOD.r14136_without_sigma_constraint_31.10.23_v0_EXT0.tap_jf17_5M_XYZ.root.tgz'
        ),
        'target': 0
    },
    'zee': {
        'datapath': Path(
            '/media/lucasbanunes/KINGSTON/data/cern-datasets/mc21_isabela_qt_no_restriction/source/signal/user.isilvafe.mc21_13p6TeV.Zee.AOD.r14136_2sigma_constraint_26.10.23_v0_EXT0.tap_zee_5M_XYZ.root.tgz'
        ),
        'target': 1
    }
}
samples_per_batch = 100000
output_dir = Path('/media/lucasbanunes/KINGSTON/data/cern-datasets/mc21_isabela_qt_no_restriction/data.parquet')
if output_dir.exists():
    logger.warning(f'Output directory {output_dir} already exists. Files may be overwritten.')
else:
    output_dir.mkdir(parents=True, exist_ok=True)

In [3]:
def is_safe_int8_cast(series: pd.Series):
    unique_values = series.unique().to_numpy()
    if np.any(np.logical_and(unique_values != 0, unique_values != 1)):
        raise ValueError(f"Series must contain only 1's and 0's for int8 casting, found {unique_values}")
    return True


INT8_CASTS = [
    'el_lhtight',
    'el_lhmedium',
    'el_lhloose',
    'el_lhvloose',
    'el_dnntight',
    'el_dnnmedium',
    'el_dnnloose',
    'mc_hasMC',
    'mc_isTruthElectronAny',
    'mc_isTruthElectronFromZ',
    'mc_isTruthElectronFromJpsiPrompt'
]

def dump_df(df, file_count, processed_rows, output_dir):
    logger.info(f'Processing Batch {file_count} (Rows {processed_rows} to {processed_rows + len(df)})')
    output_path = output_dir / f'data_{file_count:04d}.parquet'
    df['id'] = pd.Series(range(processed_rows, processed_rows + len(df)), dtype="uint64[pyarrow]")
    for col in INT8_CASTS:
        is_safe_int8_cast(df[col])
        df[col] = df[col].astype('bool[pyarrow]')
    df.to_parquet(output_path, index=False)
    return file_count + 1, processed_rows + len(df)

In [4]:
df_accumulator = pd.DataFrame()
file_count = 0
processed_rows = 0

for sample_name, sample_info in source_files.items():
    for root_tar in sample_info['datapath'].glob('*.root.tgz'):
        logger.info(f'Reading {root_tar} for sample {sample_name}')
        with TemporaryDirectory() as temp_dir:
            temp_dir_path = Path(temp_dir)
            with tarfile.open(root_tar, 'r:gz') as tar:
                tar.extractall(path=temp_dir_path)
            for root_file in temp_dir_path.glob('*.root'):
                logger.info(f'Reading {root_file} for sample {sample_name}')
                df = read_ttree_as_pdf(root_file, ttree_name='run_410000/HLT/EgammaMon/summary/events')
                df['target'] = sample_info['target']
                df['target'] = df['target'].astype('uint8[pyarrow]')
                df_accumulator = pd.concat([df_accumulator, df], ignore_index=True)
                if len(df_accumulator) >= samples_per_batch:
                    file_count, processed_rows = dump_df(
                        df_accumulator, file_count, processed_rows, output_dir
                    )
                    df_accumulator = pd.DataFrame()  # Reset accumulator

if len(df_accumulator):
    file_count, processed_rows = dump_df(
        df_accumulator, file_count, processed_rows, output_dir
    )

 [Sun, 15 Mar 2026 22:46:36] INFO Reading /media/lucasbanunes/KINGSTON/data/cern-datasets/mc21_isabela_qt_no_restriction/source/bkg/user.isilvafe.mc21_13p6TeV.JF17.AOD.r14136_without_sigma_constraint_31.10.23_v0_EXT0.tap_jf17_5M_XYZ.root.tgz/user.isilvafe.35450371._000001.XYZ.root.tgz for sample jf17
/tmp/ipykernel_1094136/3714514901.py:11: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=temp_dir_path)
 [Sun, 15 Mar 2026 22:46:36] INFO Reading /tmp/tmpm7lak3e0/output.35450371._000002.root for sample jf17
 [Sun, 15 Mar 2026 22:46:36] INFO Reading /tmp/tmpm7lak3e0/output.35450371._000010.root for sample jf17
 [Sun, 15 Mar 2026 22:46:37] INFO Reading /tmp/tmpm7lak3e0/output.35450371._000004.root for sample jf17
 [Sun, 15 Mar 2026 22:46:37] INFO Reading /tmp/tmpm7lak3e0/output.35450371._000009.root for sample jf17
 [Sun, 15 Mar 2026 22:46:37] IN

# Validate Dataset

In [5]:
output_path_glob = output_dir / 'data_*.parquet'
print(str(output_path_glob))

/media/lucasbanunes/KINGSTON/data/cern-datasets/mc21_isabela_qt_no_restriction/data.parquet/data_*.parquet


## Check Target

In [6]:
target_query = f"""
SELECT
    COALESCE(CAST(target AS VARCHAR), 'NULL') AS value,
    COUNT(*) as count
FROM read_parquet('{str(output_path_glob)}')
GROUP BY target
ORDER BY target ASC;
"""
print(target_query)


SELECT
    COALESCE(CAST(target AS VARCHAR), 'NULL') AS value,
    COUNT(*) as count
FROM read_parquet('/media/lucasbanunes/KINGSTON/data/cern-datasets/mc21_isabela_qt_no_restriction/data.parquet/data_*.parquet')
GROUP BY target
ORDER BY target ASC;



In [7]:
with duckdb.connect(':memory:') as conn:
    df = conn.execute(target_query).fetch_arrow_table().to_pandas(types_mapper=pd.ArrowDtype)
# df.value_counts(dropna=False)
df

,value,count
0,0,4988951
1,1,2177107


## Check ID

In [8]:
id_query = f"""
SELECT
    COALESCE(CAST(id AS VARCHAR), 'NULL') AS value,
    COUNT(*) as count
FROM read_parquet('{str(output_path_glob)}')
GROUP BY id
ORDER BY id ASC;
"""

In [12]:
with duckdb.connect(':memory:') as conn:
    df = conn.execute(id_query).fetch_arrow_table().to_pandas(types_mapper=pd.ArrowDtype)
# df.value_counts(dropna=False)
df

,value,count
0,0,1
1,1,1
2,2,1
3,3,1
4,4,1
...,...,...
7166053,7166053,1
7166054,7166054,1
7166055,7166055,1
7166056,7166056,1


In [13]:
df[df['count'] != 1]

,value,count


# Convert Reference

In [1]:
import json
from collections import defaultdict
import pandas as pd

In [2]:
ref_path = '/media/lucasbanunes/KINGSTON/data/cern/global-ringer/datasets/qt_data_mc21_5m/mc21_13p6TeV.Run3_v1.40bins.ref.json'

In [3]:
with open(ref_path, 'r') as f:
    ref = json.load(f)
ref

[[{'tight': {'det': {'total': 232913, 'passed': 232060},
    'fake': {'total': 1946524, 'passed': 435731}},
   'medium': {'det': {'total': 232913, 'passed': 232106},
    'fake': {'total': 1946524, 'passed': 449170}},
   'loose': {'det': {'total': 232913, 'passed': 232102},
    'fake': {'total': 1946524, 'passed': 448182}},
   'vloose': {'det': {'total': 232913, 'passed': 232150},
    'fake': {'total': 1946524, 'passed': 464211}}},
  {'tight': {'det': {'total': 73327, 'passed': 72911},
    'fake': {'total': 1912796, 'passed': 464002}},
   'medium': {'det': {'total': 73327, 'passed': 72926},
    'fake': {'total': 1912796, 'passed': 474462}},
   'loose': {'det': {'total': 73327, 'passed': 72952},
    'fake': {'total': 1912796, 'passed': 494841}},
   'vloose': {'det': {'total': 73327, 'passed': 72961},
    'fake': {'total': 1912796, 'passed': 504975}}},
  {'tight': {'det': {'total': 3205, 'passed': 3010},
    'fake': {'total': 936912, 'passed': 131386}},
   'medium': {'det': {'total': 3205

In [37]:
et_bins = [
    '3000',
    '7000',
    '10000',
    '15000',
    '20000',
    '30000',
    '40000',
    '50000',
    'inf']
eta_bins = [
    '0.0',
    '0.8',
    '1.37',
    '1.54',
    '2.37',
    '2.5'
]

In [38]:
ref_data = defaultdict(list)
iterations = 0
for iet_bin, et_refs in enumerate(ref):
    et_bin_lower = et_bins[iet_bin]
    et_bin_upper = et_bins[iet_bin + 1]
    for ieta_bin, et_eta_refs in enumerate(et_refs):
        eta_bin_lower = eta_bins[ieta_bin]
        eta_bin_upper = eta_bins[ieta_bin + 1]
        for criteria, criteria_ref in et_eta_refs.items():
            for sample_type, sample_ref in criteria_ref.items():
                for total_or_passed, value in sample_ref.items():
                    print(f'{iterations} - ET Bin: {et_bin_lower}, {et_bin_upper}, ETA Bin: {eta_bin_lower}, {eta_bin_upper}, Criteria: {criteria}, Sample Type: {sample_type}, Total/Passed: {total_or_passed}, Value: {value}')
                    ref_data['et_bin_lower'].append(et_bin_lower)
                    ref_data['et_bin_upper'].append(et_bin_upper)
                    ref_data['eta_bin_lower'].append(eta_bin_lower)
                    ref_data['eta_bin_upper'].append(eta_bin_upper)
                    ref_data['criteria'].append(criteria)
                    ref_data['sample_type'].append(sample_type)
                    ref_data['total_or_passed'].append(total_or_passed)
                    ref_data['value'].append(value)
                    iterations += 1
ref_data = pd.DataFrame(ref_data)
ref_data

0 - ET Bin: 3000, 7000, ETA Bin: 0.0, 0.8, Criteria: tight, Sample Type: det, Total/Passed: total, Value: 232913
1 - ET Bin: 3000, 7000, ETA Bin: 0.0, 0.8, Criteria: tight, Sample Type: det, Total/Passed: passed, Value: 232060
2 - ET Bin: 3000, 7000, ETA Bin: 0.0, 0.8, Criteria: tight, Sample Type: fake, Total/Passed: total, Value: 1946524
3 - ET Bin: 3000, 7000, ETA Bin: 0.0, 0.8, Criteria: tight, Sample Type: fake, Total/Passed: passed, Value: 435731
4 - ET Bin: 3000, 7000, ETA Bin: 0.0, 0.8, Criteria: medium, Sample Type: det, Total/Passed: total, Value: 232913
5 - ET Bin: 3000, 7000, ETA Bin: 0.0, 0.8, Criteria: medium, Sample Type: det, Total/Passed: passed, Value: 232106
6 - ET Bin: 3000, 7000, ETA Bin: 0.0, 0.8, Criteria: medium, Sample Type: fake, Total/Passed: total, Value: 1946524
7 - ET Bin: 3000, 7000, ETA Bin: 0.0, 0.8, Criteria: medium, Sample Type: fake, Total/Passed: passed, Value: 449170
8 - ET Bin: 3000, 7000, ETA Bin: 0.0, 0.8, Criteria: loose, Sample Type: det, Tota

,et_bin_lower,et_bin_upper,eta_bin_lower,eta_bin_upper,criteria,sample_type,total_or_passed,value
0,3000,7000,0.0,0.8,tight,det,total,232913
1,3000,7000,0.0,0.8,tight,det,passed,232060
2,3000,7000,0.0,0.8,tight,fake,total,1946524
3,3000,7000,0.0,0.8,tight,fake,passed,435731
4,3000,7000,0.0,0.8,medium,det,total,232913
...,...,...,...,...,...,...,...,...
635,50000,inf,2.37,2.5,loose,fake,passed,863
636,50000,inf,2.37,2.5,vloose,det,total,120000
637,50000,inf,2.37,2.5,vloose,det,passed,119870
638,50000,inf,2.37,2.5,vloose,fake,total,22441


In [39]:
ref_data.dtypes

et_bin_lower         str
et_bin_upper         str
eta_bin_lower        str
eta_bin_upper        str
criteria             str
sample_type          str
total_or_passed      str
value              int64
dtype: object

In [40]:
ref_data = ref_data.convert_dtypes(dtype_backend='pyarrow')
ref_data['et_bin_lower'] = ref_data['et_bin_lower'].astype('float32[pyarrow]')
ref_data['et_bin_upper'] = ref_data['et_bin_upper'].astype('float32[pyarrow]')
ref_data['eta_bin_lower'] = ref_data['eta_bin_lower'].astype('float32[pyarrow]')
ref_data['eta_bin_upper'] = ref_data['eta_bin_upper'].astype('float32[pyarrow]')
ref_data['value'] = ref_data['value'].astype('uint64[pyarrow]')
ref_data.dtypes

et_bin_lower        float[pyarrow]
et_bin_upper        float[pyarrow]
eta_bin_lower       float[pyarrow]
eta_bin_upper       float[pyarrow]
criteria           string[pyarrow]
sample_type        string[pyarrow]
total_or_passed    string[pyarrow]
value              uint64[pyarrow]
dtype: object

In [41]:
expected_ref_data_schema = pa.DataFrameSchema({
    'et_bin_lower': pa.Column(
        'float32[pyarrow]',
        checks=[
            pa.Check.ge(0)
        ]
    ),
    'et_bin_upper': pa.Column(
        'float32[pyarrow]',
        checks=[
            pa.Check.ge(0)
        ]
    ),
    'eta_bin_lower': pa.Column(
        'float32[pyarrow]',
        checks=[
            pa.Check.ge(0)
        ]
    ),
    'eta_bin_upper': pa.Column(
        'float32[pyarrow]',
        checks=[
            pa.Check.ge(0)
        ]
    ),
    'criteria': pa.Column(
        pd.ArrowDtype(pyarrow.string()),
        checks=[
            pa.Check.isin(['tight', 'medium', 'loose', 'vloose'])
        ],
    ),
    'sample_type': pa.Column(
        pd.ArrowDtype(pyarrow.string()),
        checks=[
            pa.Check.isin(['det', 'fake'])
        ]
    ),
    'total_or_passed': pa.Column(
        pd.ArrowDtype(pyarrow.string()),
        checks=[
            pa.Check.isin(['total', 'passed'])
        ]
    ),
    'value': pa.Column(
        'uint64[pyarrow]',
        checks=[
            pa.Check.ge(0)
        ]
    )
})

In [42]:
ref_data = expected_ref_data_schema.validate(ref_data)
ref_data

,et_bin_lower,et_bin_upper,eta_bin_lower,eta_bin_upper,criteria,sample_type,total_or_passed,value
0,3000.0,7000.0,0.0,0.8,tight,det,total,232913
1,3000.0,7000.0,0.0,0.8,tight,det,passed,232060
2,3000.0,7000.0,0.0,0.8,tight,fake,total,1946524
3,3000.0,7000.0,0.0,0.8,tight,fake,passed,435731
4,3000.0,7000.0,0.0,0.8,medium,det,total,232913
...,...,...,...,...,...,...,...,...
635,50000.0,inf,2.37,2.5,loose,fake,passed,863
636,50000.0,inf,2.37,2.5,vloose,det,total,120000
637,50000.0,inf,2.37,2.5,vloose,det,passed,119870
638,50000.0,inf,2.37,2.5,vloose,fake,total,22441


In [44]:
ref_data.to_parquet(output_dir.parent / 'ref.parquet')